## **Financial Month-End Closing Automation System**

In [24]:
# ==========================================
# 0. 安裝 Gemini SDK
# ==========================================
!pip install -q -U google-genai

import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
from google import genai
from google.colab import userdata

# ==========================================
# 1. 設定 Gemini API Key
# ==========================================
# Colab 左側「鑰匙圖示」新增：
# 名稱：GEMINI_API_KEY
# 值：貼上 API Key
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))


# ==========================================
# 2. 生成模擬 Excel 資料
# ==========================================
def generate_large_scale_data():
    print("正在生成模擬 Excel 檔案...")

    depts = [
        {
            "name": "Sales",
            "file": "sales_large.xlsx",
            "date_col": "Order_Date",
            "amt_col": "Revenue",
            "dept_col": "Dept_Name",
        },
        {
            "name": "行銷部",
            "file": "marketing_large.xlsx",
            "date_col": "日期",
            "amt_col": "金額",
            "dept_col": "部門",
        },
        {
            "name": "管理部",
            "file": "admin_large.xlsx",
            "date_col": "Date",
            "amt_col": "Amount",
            "dept_col": "Dept",
        },
    ]

    np.random.seed(42)

    for d in depts:
        rows = 333

        start_date = datetime(2026, 4, 1)
        random_days = np.random.randint(0, 30, size=rows)
        dates = [start_date + timedelta(days=int(i)) for i in random_days]

        # 產生一堆隨機金額，大部分集中在 20,000 附近
        amounts = np.random.normal(loc=20000, scale=15000, size=rows).astype(int)

        df = pd.DataFrame({
            d["date_col"]: dates,
            d["dept_col"]: [d["name"]] * rows,
            d["amt_col"]: amounts,
        })

        # 注入異常資料 故意讓 1% 的資料變成：負數金額、超大金額
        df.loc[df.sample(frac=0.01, random_state=1).index, d["amt_col"]] = -500
        df.loc[df.sample(frac=0.01, random_state=2).index, d["amt_col"]] = 250000
        # 行銷部還多一個異常 故意把部分資料日期改成 5 月，讓它變成「非本月資料」
        if d["name"] == "行銷部":
            df.loc[df.sample(frac=0.01, random_state=3).index, d["date_col"]] = datetime(2026, 5, 20)

        df.to_excel(d["file"], index=False)

    print("✅ 模擬 Excel 檔案生成成功！")


# ==========================================
# 3. 財務資料整併與異常檢查
# ==========================================
# 批次讀取資料夾中的 Excel，統一欄位名稱，合併成總表，再檢查異常資料
def process_finance_logic(folder_path):
    all_data = []

    # 雖然不同部門欄位名稱不一樣，但我把它們統一成：Date、Dept、Amt
    field_map = {
        "Date": ["Order_Date", "日期", "Date"],
        "Dept": ["Dept_Name", "部門", "Dept"],
        "Amt": ["Revenue", "金額", "Amount"],
    }

    inv_map = {
        old_name: standard_name
        for standard_name, old_names in field_map.items()
        for old_name in old_names
    }
    # 它會去指定資料夾找出：檔名包含 large 副檔名是 .xlsx
    files = [
        f for f in os.listdir(folder_path)
        if "large" in f and f.endswith(".xlsx")
    ]
    # 每讀一個 Excel，就做四件事：讀檔案、統一欄位名稱、加上來源檔名、只留下需要的欄位
    for file in files:
        df = pd.read_excel(os.path.join(folder_path, file))

        df = df.rename(columns=inv_map)

        df["Source"] = file

        all_data.append(df[["Date", "Dept", "Amt", "Source"]])
    # 把三個部門的 Excel 疊成一張大表。
    full_df = pd.concat(all_data, ignore_index=True)

    full_df["Amt"] = pd.to_numeric(full_df["Amt"], errors="coerce")
    full_df["Date"] = pd.to_datetime(full_df["Date"], errors="coerce")

    full_df["Check"] = ""

    full_df.loc[full_df["Amt"] < 0, "Check"] += "金額為負數; "
    full_df.loc[full_df["Amt"] > 100000, "Check"] += "超過預算閾值; "
    full_df.loc[full_df["Date"].dt.month != 4, "Check"] += "非本月資料; "
    full_df.loc[full_df["Amt"].isna(), "Check"] += "金額格式錯誤; "
    full_df.loc[full_df["Date"].isna(), "Check"] += "日期格式錯誤; "

    anomaly_df = full_df[full_df["Check"] != ""].copy()

    return full_df, anomaly_df


# ==========================================
# 4. Gemini AI 摘要
# ==========================================
def get_ai_report_with_gemini(anomaly_df):
    if anomaly_df.empty:
        return "✅ 本月財務核銷無異常，所有部門報表格式與金額均符合規範。"

    total_anomalies = len(anomaly_df)
    dept_summary = anomaly_df["Dept"].value_counts().to_string()

    # 避免資料太多，先給 AI 前 10 筆代表性異常
    anomaly_sample = anomaly_df.head(10).to_string(index=False)

    prompt = f"""
你是銀行總行財務部門實習生，請根據以下「月結異常資料」產生一份專業摘要。

請遵守：
1. 不要編造不存在的資料。
2. 只能根據我提供的異常清單摘要。
3. 語氣要像給財務部主管看的報告。
4. 請用繁體中文。
5. 請包含：
   - 總體異常概況
   - 各部門異常分布
   - 需要優先複核的項目
   - 後續處理建議

本月異常總筆數：{total_anomalies}

各部門異常筆數：
{dept_summary}

異常資料範例：
{anomaly_sample}
"""

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt
    )

    return response.text


# ==========================================
# 5. 輸出 Excel 報表
# ==========================================
def export_report(df_all, df_anomaly, ai_report):
    output_file = "finance_monthly_check_report.xlsx"

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df_all.to_excel(writer, sheet_name="整併後總表", index=False)
        df_anomaly.to_excel(writer, sheet_name="異常清單", index=False)

        report_df = pd.DataFrame({
            "AI財務摘要": ai_report.split("\n")
        })
        report_df.to_excel(writer, sheet_name="AI摘要報告", index=False)

    print(f"✅ 已輸出 Excel 報表：{output_file}")


# ==========================================
# 6. 主執行流程
# ==========================================
generate_large_scale_data()

print("正在處理資料...")
df_all, df_anomaly = process_finance_logic(".")

print(f"全部資料筆數：{len(df_all)}")
print(f"異常資料筆數：{len(df_anomaly)}")

if not df_anomaly.empty:
    print("\n--- 異常清單前幾筆 ---")
    print(df_anomaly.head())

print("\n--- 正在產生 Gemini AI 財務摘要 ---")
ai_report = get_ai_report_with_gemini(df_anomaly)

print("\n【Gemini AI 財務摘要報告】")
print(ai_report)

export_report(df_all, df_anomaly, ai_report)

正在生成模擬 Excel 檔案...
✅ 模擬 Excel 檔案生成成功！
正在處理資料...
全部資料筆數：999
異常資料筆數：102

--- 異常清單前幾筆 ---
         Date Dept     Amt            Source     Check
6  2026-04-12  管理部  -11513  admin_large.xlsx   金額為負數; 
7  2026-04-21  管理部  250000  admin_large.xlsx  超過預算閾值; 
38 2026-04-17  管理部   -5271  admin_large.xlsx   金額為負數; 
49 2026-04-25  管理部   -4656  admin_large.xlsx   金額為負數; 
52 2026-04-06  管理部   -3290  admin_large.xlsx   金額為負數; 

--- 正在產生 Gemini AI 財務摘要 ---

【Gemini AI 財務摘要報告】
致：財務部主管 / 相關管理階層
副本：財務會計處、內部稽核組
日期：2026年5月
主旨：**2026年4月份會計月結異常帳項彙總摘要報告**

關於本月（2026年4月）之月結作業，經系統自動檢核與帳務核對後，共計發現 102 筆異常項目。現就異常概況、部門分佈及後續處理建議彙整如下：

### 一、 總體異常概況
本月偵測到之 102 筆異常資料，主要集中於「金額邏輯錯誤（負數值）」以及「超額支出控管（超過預算閾值）」兩大類別。由初步抽查顯示，部分項目涉及大額異常波動，恐影響月結損益之準確性。

### 二、 各部門異常分布
本次異常案件涉及三個主要部門，分布情況如下：
1. **行銷部**：44 筆（佔比最高，需重點關注活動經費核銷情況）。
2. **管理部**：30 筆（異常集中於來源檔案 `admin_large.xlsx`）。
3. **Sales（業務部）**：28 筆。

### 三、 需要優先複核的項目
根據抽樣清單，管理部（Dept: 管理部）之異常數據呈現高度規律性錯誤，請優先針對以下項目進行人工覆核：
1. **鉅額預算超支項目**：
   - 2026-04-21 管理部產生一筆金額為 **250,000** 之帳項（來源：